# NVIDIA DLI: Image Classification Assessment

**Source:** [NVIDIA Deep Learning Institute](https://www.nvidia.com/dli)

**Description:** Assessment notebook for the NVIDIA DLI deep learning course. Train a transfer
learning model (VGG16 pretrained on ImageNet) to classify fresh and rotten fruits, achieving
at least 92% validation accuracy.

**Requirements:**
- TensorFlow / Keras
- Dataset: Fresh and Rotten Fruits from [Kaggle](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification)

**Note:** This notebook was designed for the NVIDIA DLI platform. Some cells reference
course-specific files (`run_assessment.py`, `images/` folder). Paths may need adjustment
for local execution.

## Setup

In [ ]:
# Uncomment to install dependencies if needed
# !pip install tensorflow

from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Load ImageNet Base Model

We start with a VGG16 model pretrained on ImageNet. The model is loaded with the correct weights,
an input shape of (224, 224, 3), and the top classification layer removed so we can add our own.

In [ ]:
base_model = keras.applications.VGG16(
    weights='imagenet',
    input_shape=(224, 224, 3),
    include_top=False
)

## Freeze Base Model

Freeze the base model so that all the learned features from ImageNet are not destroyed
during initial training. Only our new classification layers will be trained.

In [ ]:
# Freeze base model
base_model.trainable = False
base_model.summary()

## Add Layers to Model

Add custom classification layers on top of the pretrained base. Pay close attention
to the final Dense layer which should match the number of fruit categories (6 classes:
fresh/rotten for apples, bananas, oranges).

In [ ]:
# Create inputs with correct shape
inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

# Add pooling layer
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)

# Add final dense layer
outputs = keras.layers.Dense(6, activation='softmax')(x)

# Combine inputs and outputs to create model
model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.summary()

## Compile Model

Compile with categorical crossentropy loss (multi-class classification) and accuracy metric.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[keras.metrics.CategoricalAccuracy()]
)

## Augment the Data

Data augmentation helps improve generalization by applying random transformations to training images.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


datagen_train = ImageDataGenerator(
    samplewise_center=True,  # set each sample mean to 0
    rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
    zoom_range=0.1,  # Randomly zoom image
    width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
    height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
    horizontal_flip=True,  # randomly flip images
    vertical_flip=False,
)  # we don't expect Bo to be upside-down so we will not flip vertically


#datagen_train = ImageDataGenerator(samplewise_center=True)
datagen_valid = ImageDataGenerator(samplewise_center=True)


## Load Dataset

> **Note:** Update the directory paths below to point to your local copy of the
> [Fruits Fresh and Rotten](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification) dataset.
> The original notebook used Google Drive mounting for Colab.

In [ ]:
# --- Configure dataset paths ---
# Update these paths to your local dataset location
TRAIN_DIR = "./dataset/train"
VALID_DIR = "./dataset/valid"

# Load and iterate training dataset
train_it = datagen_train.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32,
)

# Load and iterate validation dataset
valid_it = datagen_valid.flow_from_directory(
    directory=VALID_DIR,
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32,
)

## Train the Model

Train the model using the train and validation iterators. Adjust the number of epochs as needed.

In [ ]:
model.fit(
    train_it,
    validation_data=valid_it,
    steps_per_epoch=train_it.samples / train_it.batch_size,
    validation_steps=valid_it.samples / valid_it.batch_size,
    epochs=20,
)

## Unfreeze Model for Fine Tuning

If you have reached 92% validation accuracy, this step is optional. Otherwise, fine-tune
the entire model with a very low learning rate.

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Compile with a low learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=0.00001),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[keras.metrics.CategoricalAccuracy()],
)

model.fit(
    train_it,
    validation_data=valid_it,
    steps_per_epoch=train_it.samples / train_it.batch_size,
    validation_steps=valid_it.samples / valid_it.batch_size,
    epochs=10,
)

## Evaluate the Model

Check final validation accuracy. Target: 92% or higher.

In [ ]:
model.evaluate(valid_it, steps=valid_it.samples / valid_it.batch_size)

## Run the Assessment (NVIDIA DLI Only)

> **Note:** The following cells require the `run_assessment.py` script from the NVIDIA DLI
> platform. They will not work in a standalone environment.

In [ ]:
# --- NVIDIA DLI platform only ---
# from run_assessment import run_assessment
# run_assessment(model, valid_it)